# 01 - Nsight Systems Trace-Guided Optimization Lab

This notebook is the hands-on lab portion of the workshop. In this notebook we will be working to analyze and optimize an example neural network workflow written in PyTorch. This workflow consists of functions spread out over smaller scripts we will be optimizing and iterating on. 

The intended loop will look something like the following:

1. Profile the original problem pipeline.
2. Inspect the relevant module and the reference implementation.
3. Edit the problem module.
4. Re-profile and compare the region time, throughput, and Nsight Systems timeline.


## What We Are Profiling

The workload is a synthetic PyTorch training pipeline that is intentionally ordinary-looking. Each sample starts as deterministic CPU data, goes through CPU feature engineering, moves from host memory to the selected device, then trains a dense MLP plus classifier head. The model and labels are simple, but the surrounding pipeline has the same kinds of performance patterns that show up in real training jobs.

The `problem` pipeline is the version you will debug. It was designed with four issues to remediate.

1. Extra synchronization and device-to-host metric reads in `synchronization.py`.
2. Small, blocking host-to-device transfers in `batching.py`, which increase copy overhead per sample and limit overlap.
3. Many short-lived kernel launches from micro-batched optimizer steps in `short_kernels.py`.
4. CPU preprocessing on the main training path in `handoffs.py`, which serializes input preparation with GPU work.

`scripts/run_original_pipeline.py` drives a frozen all-original baseline, and `scripts/run_solution_pipeline.py` drives a fully remediated reference used for the opening comparison. `scripts/run_problem_pipeline.py` drives the editable attendee implementation in `profiling_workshop/pipeline/problem`. After the opening comparison, `scripts/run_example_pipeline.py` drives progressive reference checkpoints: sync-only, sync plus IO, sync plus IO plus kernel launches, and finally all four Nsight Systems lab fixes. The quick `RESULT` and `REGION` lines are useful for notebook comparisons, but the Nsight Systems timeline and NVTX ranges are the source of truth for understanding what changed.


## Setup

Run this cell once at the top of the notebook, and run it again if you restart the kernel. It finds the workshop root, adds the repository to `sys.path`, creates the `traces/` output directory, and records whether CUDA and `nsys` are available.

If the selected device is CPU, the training scripts still run and the notebook still compares the printed region timers. The Nsight Systems capture cells will skip report generation because they need both CUDA and the `nsys` CLI on `PATH`.


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import re
import shutil
import subprocess
import sys
from typing import Sequence

from IPython.display import Markdown, display

try:
    import torch
except ModuleNotFoundError as exc:
    if exc.name != "torch":
        raise
    raise ModuleNotFoundError(
        "PyTorch is not installed in the active notebook kernel. "
        "In Brev, change the notebook kernel to 'Python (profiling-workshop)', "
        "or rerun brev_startup.sh so it creates the project environment."
    ) from exc

ROOT = Path.cwd()
if not (ROOT / "profiling_workshop").exists() and (ROOT.parent / "profiling_workshop").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from profiling_workshop.common import cuda_is_available_quietly

TRACE_DIR = ROOT / "traces"
TRACE_DIR.mkdir(exist_ok=True)

ORIGINAL_SCRIPT = ROOT / "scripts" / "run_original_pipeline.py"
PROBLEM_SCRIPT = ROOT / "scripts" / "run_problem_pipeline.py"
SOLUTION_SCRIPT = ROOT / "scripts" / "run_solution_pipeline.py"
EXAMPLE_SCRIPT = ROOT / "scripts" / "run_example_pipeline.py"
ORIGINAL_DIR = ROOT / "profiling_workshop" / "pipeline" / "original"
PROBLEM_DIR = ROOT / "profiling_workshop" / "pipeline" / "problem"
SOLUTION_DIR = ROOT / "profiling_workshop" / "pipeline" / "solution"

PROFILE_DEVICE = "cuda" if cuda_is_available_quietly() else "cpu"
NSYS = shutil.which("nsys")
NSYS_AVAILABLE = NSYS is not None
RUN_NSYS = PROFILE_DEVICE == "cuda" and NSYS_AVAILABLE

print(f"Workshop root: {ROOT}")
print(f"Trace directory: {TRACE_DIR}")
print(f"Profile device: {PROFILE_DEVICE}")
print(f"nsys: {NSYS or 'not found'}")
if PROFILE_DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Workload Arguments

These argument lists define the exact workload used throughout the lab. The CUDA path keeps `--samples`, model size, and the baseline CPU feature work fixed for the opening comparisons so that changes in throughput mostly come from pipeline behavior rather than from changing the amount of model work.

In [ ]:
if PROFILE_DEVICE == "cuda":
    COMMON_ARGS = [
        "--device", "cuda",
        "--samples", "8192",
        "--features", "2048",
        "--hidden", "4096",
        "--depth", "4",
        "--classes", "64",
        "--log-every", "0",
    ]
    ORIGINAL_ARGS = [
        *COMMON_ARGS,
        "--cpu-work", "2",
        "--batch-size", "128",
        "--micro-batches", "16",
        "--num-workers", "0",
        "--prefetch-batches", "0",
        "--head", "broadcast-distance",
    ]
    SYNC_FIXED_ARGS = [*ORIGINAL_ARGS]
    IO_FIXED_ARGS = [
        *COMMON_ARGS,
        "--cpu-work", "2",
        "--batch-size", "1024",
        "--micro-batches", "16",
        "--num-workers", "0",
        "--prefetch-batches", "2",
        "--head", "broadcast-distance",
    ]
    KERNEL_FIXED_ARGS = [
        *COMMON_ARGS,
        "--cpu-work", "2",
        "--batch-size", "1024",
        "--micro-batches", "128",
        "--num-workers", "0",
        "--prefetch-batches", "2",
        "--head", "broadcast-distance",
    ]
    HANDOFF_FIXED_ARGS = [
        *COMMON_ARGS,
        "--cpu-work", "6",
        "--batch-size", "1024",
        "--micro-batches", "128",
        "--num-workers", "4",
        "--prefetch-batches", "2",
        "--head", "broadcast-distance",
    ]
    SOLUTION_ARGS = [
        *COMMON_ARGS,
        "--cpu-work", "2",
        "--batch-size", "1024",
        "--micro-batches", "1",
        "--num-workers", "4",
        "--prefetch-batches", "2",
        "--head", "matmul-distance",
    ]
else:
    COMMON_ARGS = [
        "--device", "cpu",
        "--samples", "256",
        "--features", "64",
        "--hidden", "128",
        "--depth", "2",
        "--classes", "8",
        "--log-every", "0",
    ]
    ORIGINAL_ARGS = [*COMMON_ARGS, "--cpu-work", "1", "--batch-size", "32", "--micro-batches", "4", "--num-workers", "0", "--prefetch-batches", "0"]
    SYNC_FIXED_ARGS = [*ORIGINAL_ARGS]
    IO_FIXED_ARGS = [*COMMON_ARGS, "--cpu-work", "1", "--batch-size", "64", "--micro-batches", "4", "--num-workers", "0", "--prefetch-batches", "0"]
    KERNEL_FIXED_ARGS = [*COMMON_ARGS, "--cpu-work", "1", "--batch-size", "64", "--micro-batches", "32", "--num-workers", "0", "--prefetch-batches", "0"]
    HANDOFF_FIXED_ARGS = [*COMMON_ARGS, "--cpu-work", "2", "--batch-size", "64", "--micro-batches", "32", "--num-workers", "0", "--prefetch-batches", "0"]
    SOLUTION_ARGS = [*COMMON_ARGS, "--cpu-work", "1", "--batch-size", "64", "--micro-batches", "1", "--num-workers", "0", "--prefetch-batches", "0"]

PROBLEM_ARGS = ORIGINAL_ARGS
PROGRESSION_ARGS = {
    "original": ORIGINAL_ARGS,
    "sync": SYNC_FIXED_ARGS,
    "io": IO_FIXED_ARGS,
    "kernels": KERNEL_FIXED_ARGS,
    "handoff": HANDOFF_FIXED_ARGS,
}

PROGRESSION_ARGS, SOLUTION_ARGS


## Helpers

This first helper cell is notebook plumbing. It runs scripts, parses their `RESULT` and `REGION` lines, renders quick comparison tables, and points you to the editable problem file plus the matching reference file.

You do not need to edit this helper code during the lab. When a later cell says to inspect a module, open the listed files in the IDE or Jupyter file browser instead of reading a large diff in notebook output.


In [ ]:
RESULT_RE = re.compile(
    r"RESULT pipeline=(?P<pipeline>\w+) device=(?P<device>\S+) samples=(?P<samples>\d+) "
    r"seconds=(?P<seconds>[0-9.]+) samples_per_second=(?P<sps>[0-9.]+) "
    r"loss=(?P<loss>[0-9.]+) accuracy=(?P<accuracy>[0-9.]+)"
)
REGION_RE = re.compile(
    r"REGION pipeline=(?P<pipeline>\w+) name=(?P<name>\S+) "
    r"seconds=(?P<seconds>[0-9.]+) percent_of_wall=(?P<percent>[0-9.]+)"
)


def run_command(cmd: Sequence[str], check: bool = True, capture: bool = False) -> subprocess.CompletedProcess[str]:
    cmd = [str(part) for part in cmd]
    print("\n$ " + " ".join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=capture)


def parse_workload_output(output: str) -> dict[str, object]:
    result_match = RESULT_RE.search(output)
    if result_match is None:
        raise ValueError("The workload did not print a RESULT line.")
    result = result_match.groupdict()
    regions = {
        match.group("name"): {
            "seconds": float(match.group("seconds")),
            "percent": float(match.group("percent")),
        }
        for match in REGION_RE.finditer(output)
    }
    return {
        "pipeline": result["pipeline"],
        "samples": int(result["samples"]),
        "seconds": float(result["seconds"]),
        "samples_per_second": float(result["sps"]),
        "loss": float(result["loss"]),
        "accuracy": float(result["accuracy"]),
        "regions": regions,
        "output": output,
    }


def run_workload(label: str, script: Path, args: Sequence[str]) -> dict[str, object]:
    completed = run_command([sys.executable, script, *args], check=True, capture=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    parsed = parse_workload_output(completed.stdout)
    parsed["label"] = label
    return parsed


def comparison_table(problem: dict[str, object], solution: dict[str, object], region: str | None = None) -> None:
    left_label = str(problem.get("label", "left"))
    right_label = str(solution.get("label", "right"))
    rows = [
        f"| metric | {left_label} | {right_label} | speedup |",
        "|---|---:|---:|---:|",
    ]
    p_sps = problem["samples_per_second"]
    s_sps = solution["samples_per_second"]
    rows.append(f"| throughput samples/s | {p_sps:,.1f} | {s_sps:,.1f} | {s_sps / max(p_sps, 1e-12):.2f}x |")
    rows.append(f"| wall seconds | {problem['seconds']:.3f} | {solution['seconds']:.3f} | {problem['seconds'] / max(solution['seconds'], 1e-12):.2f}x faster |")
    if region is not None:
        p_region = problem["regions"].get(region, {"seconds": 0.0})["seconds"]
        s_region = solution["regions"].get(region, {"seconds": 0.0})["seconds"]
        rows.append(f"| {region} seconds | {p_region:.6f} | {s_region:.6f} | {p_region / max(s_region, 1e-12):.2f}x faster |")
    display(Markdown("\n".join(rows)))


def notebook_file_link(path: Path) -> str:
    relative = path.relative_to(ROOT).as_posix()
    return f"[`{relative}`](../{relative})"


def show_edit_targets(problem_file: str, title: str) -> None:
    problem_path = PROBLEM_DIR / problem_file
    solution_path = SOLUTION_DIR / problem_file
    rows = [
        f"### {title}",
        "",
        f"- Edit: {notebook_file_link(problem_path)}",
        f"- Reference: {notebook_file_link(solution_path)}",
    ]
    display(Markdown("\n".join(rows)))


## Nsight Systems Capture Helpers

Run this second helper cell before any trace capture. It builds the `nsys profile` command used later in the notebook, chooses a GPU metrics device when Nsight reports one, removes stale trace files before each capture, and prints a compact `nsys stats` summary after each report is generated.

For Brev, `NSYS_USE_SUDO` defaults to `1` so the GPU metrics probe and profile captures run as `sudo -n -E nsys ...` when elevated permission is needed for NVIDIA counters. Set `NSYS_USE_SUDO=0` before a capture cell to run Nsight Systems without sudo.

The generated `.nsys-rep` files land in `traces/`. The compact text table shows only CUDA API timing; the main learning happens when you open the report in Nsight Systems and line up CUDA API calls, GPU kernels, memory copies, CPU activity, and the NVTX ranges named `issue_1_*` through `issue_4_*`.


In [ ]:
NSYS_TRACE = os.environ.get("NSYS_TRACE", "cuda,nvtx,osrt,cublas,cudnn")
NSYS_EXTRA_ARGS = [
    f"--trace={NSYS_TRACE}",
    "--sample=process-tree",
    "--cpuctxsw=process-tree",
    "--backtrace=fp",
    "--cuda-memory-usage=true",
    "--stats=false",
]
NSYS_GPU_METRICS_DEVICE = os.environ.get("NSYS_GPU_METRICS_DEVICE", "auto")
NSYS_GPU_METRICS_FREQUENCY = os.environ.get("NSYS_GPU_METRICS_FREQUENCY", "10000")
NSYS_COUNTER_PERMISSION_ERROR = "ERR_NVGPUCTRPERM"
SUDO_ERROR_MARKERS = (
    "a password is required",
    "a terminal is required",
    "must have a tty",
    "not in the sudoers file",
)
_GPU_METRICS_DEVICE_CACHE: str | None | bool = False


def env_flag(name: str, *, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


def explain_nsys_sudo_unavailable(reason: str) -> None:
    display(Markdown(
        "**Nsight Systems sudo mode is enabled, but sudo could not run non-interactively.**\n\n"
        f"Reason: `{reason}`\n\n"
        "The helper uses `sudo -n -E` so the notebook does not hang waiting for a password. "
        "Run `sudo -v` in a Brev terminal first, use an environment with passwordless sudo, "
        "or set `NSYS_USE_SUDO=0` and run without sudo."
    ))


def explain_nsys_counter_permissions() -> None:
    display(Markdown(
        "**Nsight Systems could not access GPU performance counters.**\n\n"
        "The trace may still be useful, but GPU hardware metrics require permission from "
        "the NVIDIA driver or hosting platform. On Brev, this notebook defaults to "
        "`sudo -n -E nsys ...`; if that still fails, the provider is not granting the "
        "required counter access for this instance."
    ))


def with_optional_nsys_sudo(base_cmd: Sequence[str]) -> list[str] | None:
    if not env_flag("NSYS_USE_SUDO", default=True):
        return [str(part) for part in base_cmd]
    sudo = shutil.which("sudo")
    if sudo is None:
        explain_nsys_sudo_unavailable("sudo was not found on PATH")
        return None
    return [sudo, "-n", "-E", *[str(part) for part in base_cmd]]


def restore_nsys_output_ownership(report_base: Path) -> None:
    if not env_flag("NSYS_USE_SUDO", default=True):
        return
    sudo = shutil.which("sudo")
    if sudo is None:
        return
    outputs = [
        report_base.with_suffix(suffix)
        for suffix in [".nsys-rep", ".sqlite", ".qdstrm"]
        if report_base.with_suffix(suffix).exists()
    ]
    if not outputs:
        return
    subprocess.run(
        [sudo, "-n", "chown", f"{os.getuid()}:{os.getgid()}", *[str(path) for path in outputs]],
        check=False,
        text=True,
        capture_output=True,
    )


def detect_gpu_metrics_device() -> str | None:
    global _GPU_METRICS_DEVICE_CACHE
    if _GPU_METRICS_DEVICE_CACHE is not False:
        return _GPU_METRICS_DEVICE_CACHE
    if not RUN_NSYS:
        _GPU_METRICS_DEVICE_CACHE = None
        return None
    requested = NSYS_GPU_METRICS_DEVICE.lower()
    if requested in {"", "none", "off", "false"}:
        _GPU_METRICS_DEVICE_CACHE = None
        return None
    if requested != "auto":
        _GPU_METRICS_DEVICE_CACHE = NSYS_GPU_METRICS_DEVICE
        return NSYS_GPU_METRICS_DEVICE

    probe_cmd = with_optional_nsys_sudo([
        NSYS or "nsys", "profile", "--gpu-metrics-devices=help", "--duration=1", "--trace=none", "true"
    ])
    if probe_cmd is None:
        _GPU_METRICS_DEVICE_CACHE = None
        return None

    completed = subprocess.run(
        probe_cmd,
        text=True,
        capture_output=True,
        check=False,
    )
    help_text = completed.stdout + completed.stderr
    lowered_help = help_text.lower()
    if env_flag("NSYS_USE_SUDO", default=True) and any(marker in lowered_help for marker in SUDO_ERROR_MARKERS):
        explain_nsys_sudo_unavailable(help_text.strip() or f"sudo exited with return code {completed.returncode}")
        _GPU_METRICS_DEVICE_CACHE = None
        return None
    for line in help_text.splitlines():
        text = line.strip()
        if not text or text.lower().startswith("possible"):
            continue
        if "not found" in text.lower() or "not available" in text.lower():
            continue
        if text == "all" or text.startswith("all "):
            _GPU_METRICS_DEVICE_CACHE = "all"
            return "all"
        match = re.match(r"^(\d+)(?:\s|:|$)", text)
        if match:
            _GPU_METRICS_DEVICE_CACHE = match.group(1)
            return _GPU_METRICS_DEVICE_CACHE

    print("GPU HW metrics were not enabled because Nsight did not report a supported metrics device.")
    _GPU_METRICS_DEVICE_CACHE = None
    return None


def remove_stale_outputs(report_base: Path) -> None:
    for suffix in [".nsys-rep", ".sqlite", ".qdstrm"]:
        candidate = report_base.with_suffix(suffix)
        if candidate.exists():
            candidate.unlink()


def resolve_report(report_base: Path) -> Path | None:
    report = report_base.with_suffix(".nsys-rep")
    if report.exists():
        return report
    sqlite = report_base.with_suffix(".sqlite")
    if sqlite.exists():
        return sqlite
    return None


def checkpoint_args(checkpoint: str, args: Sequence[str]) -> list[str]:
    return ["--checkpoint", checkpoint, *args]


def profile_workload(name: str, script: Path, args: Sequence[str]) -> Path | None:
    if not RUN_NSYS:
        print("Skipping Nsight Systems capture because CUDA or nsys is unavailable.")
        return None
    report_base = TRACE_DIR / name
    remove_stale_outputs(report_base)
    base_cmd = [
        NSYS or "nsys",
        "profile",
        "--force-overwrite=true",
        *NSYS_EXTRA_ARGS,
    ]
    metrics_device = detect_gpu_metrics_device()
    if metrics_device is not None:
        base_cmd.extend([
            f"--gpu-metrics-devices={metrics_device}",
            f"--gpu-metrics-frequency={NSYS_GPU_METRICS_FREQUENCY}",
        ])
        print(f"GPU metrics enabled for device setting: {metrics_device}")
    base_cmd.extend([
        "-o", str(report_base),
        sys.executable,
        str(script),
        *args,
    ])
    cmd = with_optional_nsys_sudo(base_cmd)
    if cmd is None:
        return None
    completed = run_command(cmd, check=False, capture=True)
    output = (completed.stdout or "") + (completed.stderr or "")
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    restore_nsys_output_ownership(report_base)
    report = resolve_report(report_base)
    if report is not None:
        print(f"Nsight report: {report}")
    elif env_flag("NSYS_USE_SUDO", default=True) and any(marker in output.lower() for marker in SUDO_ERROR_MARKERS):
        explain_nsys_sudo_unavailable(output.strip() or f"sudo exited with return code {completed.returncode}")
    elif NSYS_COUNTER_PERMISSION_ERROR in output:
        explain_nsys_counter_permissions()
    elif completed.returncode != 0:
        print(f"nsys exited with return code {completed.returncode}")
    return report


def print_limited_output(output: str, *, max_lines: int = 32) -> None:
    lines = [line for line in output.splitlines() if line.strip()]
    if not lines:
        return
    shown = lines[:max_lines]
    print("\n".join(shown))
    if len(lines) > max_lines:
        print(f"... {len(lines) - max_lines} more lines omitted. Open the report for full details.")


def nsys_stats(report: Path | None) -> None:
    if report is None:
        print("No Nsight Systems report to summarize.")
        return
    if shutil.which("nsys") is None:
        print("nsys was not found on PATH.")
        return
    cmd = [
        NSYS or "nsys",
        "stats",
        "--force-export=true",
        "--report", "cuda_api_sum",
        "--format", "table",
        str(report),
    ]
    completed = run_command(cmd, check=False, capture=True)
    output = (completed.stdout or "") + (completed.stderr or "")
    print("CUDA API timing summary:")
    print_limited_output(output, max_lines=int(os.environ.get("NSYS_SUMMARY_LINES", "32")))


def profile_reference_pair() -> tuple[Path | None, Path | None]:
    original_report = profile_workload("reference_all_original", ORIGINAL_SCRIPT, ORIGINAL_ARGS)
    solution_report = profile_workload("reference_all_fixed", SOLUTION_SCRIPT, SOLUTION_ARGS)
    return original_report, solution_report


def profile_checkpoint(issue_name: str, checkpoint: str, args: Sequence[str]) -> tuple[Path | None, Path | None]:
    user_report = profile_workload(f"{issue_name}_user", PROBLEM_SCRIPT, args)
    example_report = profile_workload(f"{issue_name}_example", EXAMPLE_SCRIPT, checkpoint_args(checkpoint, args))
    return user_report, example_report


## Original Outputs

Run both pipelines once before making edits. Keep these numbers as the starting point for the lab; they tell you whether your later changes moved the right needle.

The frozen `original` and `solution` scripts process the same number of samples with the same model dimensions, but their pipeline choices differ. Expect the reference solution to have higher throughput and lower time in the four named regions. If both runs are extremely short, increase `--samples` in the workload argument cell before collecting traces.

In [ ]:
original_problem = run_workload("all original", ORIGINAL_SCRIPT, ORIGINAL_ARGS)
reference_solution = run_workload("all fixed reference", SOLUTION_SCRIPT, SOLUTION_ARGS)
comparison_table(original_problem, reference_solution)


Save the opening reference profiles before you start editing. These two reports are the bookends for the lab: the frozen all-original implementation and the fully fixed reference implementation.


In [ ]:
reference_original_report, reference_solution_report = profile_reference_pair()
nsys_stats(reference_original_report)
nsys_stats(reference_solution_report)


## 1. Remove Extra Synchronization

### First target file for attendees to edit: `profiling_workshop/pipeline/problem/synchronization.py`

CUDA kernels and copies are normally queued asynchronously, which lets the CPU prepare future work while the GPU is still running. This problem path interrupts that flow by synchronizing after H2D transfer and by pulling metrics back to the CPU inside every training step.

In Nsight Systems, look for CUDA API rows such as `cudaDeviceSynchronize`, D2H copies attached to metric logging, CPU gaps near metric code, and NVTX ranges beginning with `issue_1_`. The script’s printed `REGION` summary should also show non-trivial time spent in issue_1_synchronization before you make the fix.

HINT: Start in `MetricTracker.update` and `synchronize_phase`. Ask which values truly need to become Python scalars during the loop, and which can remain as device tensors until `finalize`. Removing an explicit synchronize is usually safe when later CUDA work has real tensor dependencies; CUDA will preserve those dependencies without forcing the host thread to wait early.


In [ ]:
show_edit_targets("synchronization.py", "Synchronization edit targets")

Run the script after your synchronization edit. This compares your editable pipeline against the sync-only reference checkpoint using the same workload arguments. The region-specific row compares `issue_1_synchronization`, so it should be the first place you look; total throughput may move less dramatically until the later pipeline issues are addressed.


In [ ]:
sync_problem = run_workload("your sync fix", PROBLEM_SCRIPT, SYNC_FIXED_ARGS)
sync_example = run_workload("example sync fix", EXAMPLE_SCRIPT, checkpoint_args("sync", SYNC_FIXED_ARGS))
comparison_table(sync_problem, sync_example, "issue_1_synchronization")


Capture a trace for the synchronization issue once the quick run looks sane. In the Nsight Systems UI, compare your sync-fix report with the sync-only reference checkpoint around the `issue_1_*` ranges, then use the compact CUDA API timing summary below as a quick check for fewer synchronizing API calls.


In [ ]:
sync_user_report, sync_example_report = profile_checkpoint("issue_1_sync", "sync", SYNC_FIXED_ARGS)
nsys_stats(sync_user_report)
nsys_stats(sync_example_report)


## 2. Reduce Host/Device IO Frequency

### Second target file for attendees to edit: `profiling_workshop/pipeline/problem/batching.py`

The next largest end-to-end bottleneck is the input copy path. The original path uses small batches, pageable host memory, and blocking H2D copies. That means the pipeline pays copy setup overhead more often, and the training stream tends to wait for the next batch to arrive before it can launch more kernels.

In Nsight Systems, inspect the `issue_2_batch_io_*` range together with CUDA memory-copy rows. You are looking for many small H2D transfers, host-side blocking around `.to(device)`, and poor overlap between copies and compute.

HINT: There are two levels of fix here. First, make the DataLoader use pinned memory on CUDA and use `non_blocking=True` when moving tensors to the device. Then, for the full reference-style fix, move H2D copies onto a side CUDA stream, record an event after each copy, make the current training stream wait for that event, and call `record_stream` so PyTorch does not recycle copied tensors while kernels still use them. The queue size is controlled by `config.prefetch_batches`.


In [ ]:
show_edit_targets("batching.py", "Batching and IO edit targets")


Run the workload after your batching/IO edit. This compares your editable pipeline against the reference checkpoint with synchronization and IO fixes applied. The `issue_2_batch_io` row should be close to the example, and total throughput should move visibly because larger batches and overlapped copies affect the whole training loop. The short-kernel and handoff issues are still present here, so do not expect this checkpoint to look fully optimized yet.


In [ ]:
io_problem = run_workload("your sync + IO fixes", PROBLEM_SCRIPT, IO_FIXED_ARGS)
io_example = run_workload("example sync + IO fixes", EXAMPLE_SCRIPT, checkpoint_args("io", IO_FIXED_ARGS))
comparison_table(io_problem, io_example, "issue_2_batch_io")


Capture the IO trace and compare H2D copy timing against GPU kernels in your run and the matching example. The strongest version of the fix should show fewer copies per sample, non-blocking transfer behavior from pinned host memory, and copy work queued ahead of compute rather than sitting entirely on the critical path.


In [ ]:
io_user_report, io_example_report = profile_checkpoint("issue_2_io", "io", IO_FIXED_ARGS)
nsys_stats(io_user_report)
nsys_stats(io_example_report)


## 3. Increase Useful Work per Launch

### Third target file for attendees to edit: `profiling_workshop/pipeline/problem/short_kernels.py`

After synchronization and IO are out of the way, the launch pattern is easier to see. The original path splits each batch into many micro-batches and runs a full optimizer step for each chunk. That makes the GPU do useful math, but it also creates many small launch clusters where the fixed overhead of launching kernels is large compared with the kernel execution time.

In Nsight Systems, the `issue_3_short_lived_microbatch_train_step` range should appear many times per input batch. This section deliberately uses a high `--micro-batches` value so the pattern is visible even after the IO stage introduced larger batches. Zoom into one cluster and compare the CPU launch activity with the GPU kernel durations; the teaching target is the ratio of overhead to useful work.

HINT: Look for the `torch.chunk` loop. A good first fix is to run one forward pass, loss, backward pass, optimizer step, and metric update over the whole `x, y` batch. Keep the autocast/scaler flow intact, but think carefully about where `optimizer.zero_grad`, `scaler.step`, `scaler.update`, and `metric_tracker.update` belong when there is only one training step per batch.


In [ ]:
show_edit_targets("short_kernels.py", "Short-kernel edit targets")

Run the workload again after your kernel-launch edit. This compares your editable pipeline against the reference checkpoint with synchronization, IO, and kernel-launch fixes applied. The `issue_3_short_lived_kernels` row should be close to the example because there are fewer tiny training steps; the loss and accuracy do not need to match bit-for-bit, but they should remain finite and plausible.


In [ ]:
kernel_problem = run_workload("your sync + IO + kernel fixes", PROBLEM_SCRIPT, KERNEL_FIXED_ARGS)
kernel_example = run_workload("example sync + IO + kernel fixes", EXAMPLE_SCRIPT, checkpoint_args("kernels", KERNEL_FIXED_ARGS))
comparison_table(kernel_problem, kernel_example, "issue_3_short_lived_kernels")


Capture the trace and inspect the `issue_3_*` ranges in your run and the matching example run. A successful fix should show fewer repeated launch clusters and larger chunks of work per batch. Open the full `.nsys-rep` report to compare the kernel rows; the notebook output stays focused on CUDA API timing.


In [ ]:
kernel_user_report, kernel_example_report = profile_checkpoint("issue_3_kernels", "kernels", KERNEL_FIXED_ARGS)
nsys_stats(kernel_user_report)
nsys_stats(kernel_example_report)


## 4. Overlap CPU and GPU Work

### Fourth target file for attendees to edit: `profiling_workshop/pipeline/problem/handoffs.py`

The final issue is CPU preprocessing on the main training path. The original code prepares per-sample CPU features immediately before copying each batch to the device. That creates a CPU-only gap on the critical path: while the host thread is transforming data, it is not launching GPU work for the current batch or preparing transfers for the next one.

In Nsight Systems, look for `issue_4_cpu_gpu_handoff_main_thread_preprocessing` on the CPU side and notice whether GPU work is idle nearby. This section increases `--cpu-work` and enables DataLoader workers so the gap is large enough to reason about. The goal is to move expensive, batch-independent CPU preparation to the input side of the pipeline so workers can prepare future batches while the GPU trains.

HINT: Compare `RawSyntheticDataset` and `AugmentedSyntheticDataset` in `profiling_workshop/pipeline/shared.py`. `make_dataset` decides what each DataLoader worker returns, while `prepare_batch` runs in the training path. Try to make `prepare_batch` cheap: unpack already-prepared tensors, preserve contiguity and dtype expectations, and leave the expensive feature engineering to the dataset.


In [ ]:
show_edit_targets("handoffs.py", "CPU/GPU handoff edit targets")

Run the workload after changing the handoff boundary. This compares your editable pipeline against the reference checkpoint with all four fixes applied. The `issue_4_cpu_gpu_handoff` timer should be close to the example because the training loop no longer performs the expensive CPU transformation inline. If you are running on CPU or with `--num-workers 0`, the overlap benefit will be smaller, but the module should still produce the same `(x, y)` batch shape.


In [ ]:
handoff_problem = run_workload("your all four fixes", PROBLEM_SCRIPT, HANDOFF_FIXED_ARGS)
handoff_example = run_workload("example all four fixes", EXAMPLE_SCRIPT, checkpoint_args("handoff", HANDOFF_FIXED_ARGS))
comparison_table(handoff_problem, handoff_example, "issue_4_cpu_gpu_handoff")


Capture the handoff trace and compare your CPU lane, DataLoader worker activity, and GPU queue against the matching example. The healthy shape is not just lower CPU time; it is better overlap, with preprocessing happening ahead of the batch that the GPU is currently training on.


In [ ]:
handoff_user_report, handoff_example_report = profile_checkpoint("issue_4_handoff", "handoff", HANDOFF_FIXED_ARGS)
nsys_stats(handoff_user_report)
nsys_stats(handoff_example_report)

## Final Check

After all edits, run the problem pipeline one more time and compare it against the all-four-fixes example checkpoint. A successful workshop fix should not need to match the reference exactly, but it should move in the same direction: fewer synchronization barriers, less copy overhead per sample, fewer tiny launch clusters, and better CPU/GPU overlap.

Use this final table as a quick numerical wrap-up, then return to the best final Nsight Systems report for the real conclusion. The most important skill from this lab is connecting a timeline pattern to a small, targeted code change.


In [ ]:
final_problem = run_workload("final edited problem", PROBLEM_SCRIPT, HANDOFF_FIXED_ARGS)
final_example = run_workload("example all four fixes", EXAMPLE_SCRIPT, checkpoint_args("handoff", HANDOFF_FIXED_ARGS))
comparison_table(final_problem, final_example)
